In [1]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/nassau_candy_enriched.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'])

print(f"Enriched data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Enriched data loaded: 6013 rows, 28 columns


### 1. Factory Coordinates

In [3]:
# Provided in project brief — added here for geo analysis

factory_coords = {
    "Lot's O' Nuts"    : {"latitude": 32.881893, "longitude": -111.768036, "state": "Arizona"},
    "Wicked Choccy's"  : {"latitude": 32.076176, "longitude": -81.088371,  "state": "Georgia"},
    "Sugar Shack"      : {"latitude": 48.119140, "longitude": -96.181150,  "state": "Minnesota"},
    "Secret Factory"   : {"latitude": 41.446333, "longitude": -90.565487,  "state": "Illinois"},
    "The Other Factory": {"latitude": 35.117500, "longitude": -89.971107,  "state": "Tennessee"}
}

print("FACTORY LOCATIONS")
for factory, info in factory_coords.items():
    print(f"  {factory:<20} | {info['state']:<12} | "
          f"Lat: {info['latitude']}  Lon: {info['longitude']}")

FACTORY LOCATIONS
  Lot's O' Nuts        | Arizona      | Lat: 32.881893  Lon: -111.768036
  Wicked Choccy's      | Georgia      | Lat: 32.076176  Lon: -81.088371
  Sugar Shack          | Minnesota    | Lat: 48.11914  Lon: -96.18115
  Secret Factory       | Illinois     | Lat: 41.446333  Lon: -90.565487
  The Other Factory    | Tennessee    | Lat: 35.1175  Lon: -89.971107


### 2. Factory Performance Summary

In [4]:
total_revenue = df['Sales'].sum()
total_profit  = df['Gross Profit'].sum()
company_avg_margin = (total_profit / total_revenue * 100)

factory_summary = df.groupby('Factory').agg(
    Total_Revenue       = ('Sales', 'sum'),
    Total_Profit        = ('Gross Profit', 'sum'),
    Total_Cost          = ('Cost', 'sum'),
    Total_Units         = ('Units', 'sum'),
    Num_Products        = ('Product Name', 'nunique'),
    Avg_Margin          = ('Gross_Margin_%', 'mean'),
    Avg_Profit_per_Unit = ('Profit_per_Unit', 'mean'),
    Avg_Cost_per_Unit   = ('Cost_per_Unit', 'mean')
).round(2).reset_index()

factory_summary['Revenue_Share_%'] = (factory_summary['Total_Revenue'] / total_revenue * 100).round(2)
factory_summary['Profit_Share_%']  = (factory_summary['Total_Profit']  / total_profit  * 100).round(2)
factory_summary['Profit_Efficiency'] = (
    factory_summary['Total_Profit'] / factory_summary['Total_Revenue']
).round(4)

# Margin gap vs company average — key diagnostic metric
factory_summary['Margin_vs_Company_Avg'] = (
    factory_summary['Avg_Margin'] - company_avg_margin
).round(2)

factory_summary = factory_summary.sort_values('Avg_Margin', ascending=False).reset_index(drop=True)

print(f"Company Average Margin: {company_avg_margin:.2f}%")
print()
print("FACTORY PERFORMANCE SUMMARY")
print(factory_summary[['Factory', 'Total_Revenue', 'Total_Profit', 'Avg_Margin',
                        'Margin_vs_Company_Avg', 'Profit_Share_%']].to_string(index=False))

Company Average Margin: 65.96%

FACTORY PERFORMANCE SUMMARY
          Factory  Total_Revenue  Total_Profit  Avg_Margin  Margin_vs_Company_Avg  Profit_Share_%
    Lot's O' Nuts       45394.09      31393.09       69.20                   3.24           56.78
  Wicked Choccy's       32179.25      20960.19       65.12                  -0.84           37.91
      Sugar Shack         141.34         76.14       52.09                 -13.87            0.14
   Secret Factory        5418.75       2754.95       51.71                 -14.25            4.98
The Other Factory         694.00        107.00       12.88                 -53.08            0.19


### 3. Is a Low Margin Product Low Because of Its Factory?

In [5]:
# Core question from the brief
# Compare each product's margin vs its factory's average margin

product_factory = df.groupby(['Product Name', 'Factory', 'Division']).agg(
    Total_Revenue = ('Sales', 'sum'),
    Total_Profit  = ('Gross Profit', 'sum'),
    Avg_Margin    = ('Gross_Margin_%', 'mean'),
    Total_Units   = ('Units', 'sum')
).round(2).reset_index()

# Merge factory average margin in
factory_avg = factory_summary[['Factory', 'Avg_Margin']].rename(
    columns={'Avg_Margin': 'Factory_Avg_Margin'}
)
product_factory = product_factory.merge(factory_avg, on='Factory', how='left')
product_factory['Margin_vs_Factory_Avg'] = (
    product_factory['Avg_Margin'] - product_factory['Factory_Avg_Margin']
).round(2)
product_factory['Margin_vs_Company_Avg'] = (
    product_factory['Avg_Margin'] - company_avg_margin
).round(2)

product_factory = product_factory.sort_values('Avg_Margin', ascending=False).reset_index(drop=True)

print("PRODUCT MARGIN vs FACTORY AVERAGE")
print(f"{'Product':<35} {'Factory':<20} {'Product Margin':>15} "
      f"{'Factory Avg':>12} {'Diff':>8}")
print("-" * 95)
for _, row in product_factory.iterrows():
    flag = " ⚠️" if row['Margin_vs_Factory_Avg'] < -5 else ""
    print(f"  {row['Product Name']:<33} {row['Factory']:<20} "
          f"{row['Avg_Margin']:>13.1f}%  "
          f"{row['Factory_Avg_Margin']:>10.1f}%  "
          f"{row['Margin_vs_Factory_Avg']:>+7.1f}%{flag}")

PRODUCT MARGIN vs FACTORY AVERAGE
Product                             Factory               Product Margin  Factory Avg     Diff
-----------------------------------------------------------------------------------------------
  Everlasting Gobstopper            Secret Factory                80.0%        51.7%    +28.3%
  Hair Toffee                       The Other Factory             77.8%        12.9%    +64.9%
  Wonka Bar - Nutty Crunch Surprise Lot's O' Nuts                 71.3%        69.2%     +2.1%
  Wonka Bar -Scrumdiddlyumptious    Lot's O' Nuts                 69.4%        69.2%     +0.2%
  Wonka Bar - Fudge Mallows         Lot's O' Nuts                 66.7%        69.2%     -2.5%
  Wonka Bar - Triple Dazzle Caramel Wicked Choccy's               65.3%        65.1%     +0.2%
  Wonka Bar - Milk Chocolate        Wicked Choccy's               64.9%        65.1%     -0.2%
  Laffy Taffy                       Sugar Shack                   62.3%        52.1%    +10.2%
  Fizzy Lifting

### 4. Factory Cost Structure Diagnostic

In [6]:
# High cost per unit = factory cost problem
# Compare cost structure across factories

print("FACTORY COST STRUCTURE")
print(f"{'Factory':<20} {'Avg Cost/Unit':>14} {'Avg Profit/Unit':>16} "
      f"{'Avg Margin':>11} {'Verdict':>20}")
print("-" * 85)

for _, row in factory_summary.sort_values('Avg_Cost_per_Unit', ascending=False).iterrows():
    if row['Avg_Margin'] < company_avg_margin - 10:
        verdict = "🔴 Cost Problem"
    elif row['Avg_Margin'] < company_avg_margin:
        verdict = "⚠️  Below Average"
    else:
        verdict = "✅ Healthy"

    print(f"  {row['Factory']:<18} ${row['Avg_Cost_per_Unit']:>12.2f}  "
          f"${row['Avg_Profit_per_Unit']:>13.2f}  "
          f"{row['Avg_Margin']:>9.1f}%  {verdict:>20}")

print(f"\n  Company Avg Margin: {company_avg_margin:.2f}%")

FACTORY COST STRUCTURE
Factory               Avg Cost/Unit  Avg Profit/Unit  Avg Margin              Verdict
-------------------------------------------------------------------------------------
  Secret Factory     $        4.94  $         5.10       51.7%        🔴 Cost Problem
  The Other Factory  $        2.85  $         0.49       12.9%        🔴 Cost Problem
  Wicked Choccy's    $        1.22  $         2.28       65.1%     ⚠️  Below Average
  Lot's O' Nuts      $        1.10  $         2.47       69.2%             ✅ Healthy
  Sugar Shack        $        0.90  $         1.06       52.1%        🔴 Cost Problem

  Company Avg Margin: 65.96%


### 5. Factory Risk Classification

In [7]:
def classify_factory(margin, company_avg):
    if margin >= company_avg + 5:
        return "BEST IN CLASS 🏆"
    elif margin >= company_avg:
        return "PERFORMING ✅"
    elif margin >= company_avg - 15:
        return "UNDERPERFORMING ⚠️"
    else:
        return "CRITICAL RISK 🔴"

factory_summary['Risk_Class'] = factory_summary['Avg_Margin'].apply(
    lambda m: classify_factory(m, company_avg_margin)
)

print("FACTORY RISK CLASSIFICATION")
print(f"  Company Avg Margin: {company_avg_margin:.2f}%\n")
for _, row in factory_summary.iterrows():
    print(f"  {row['Factory']:<20} | Margin: {row['Avg_Margin']:>6.2f}% "
          f"| Gap: {row['Margin_vs_Company_Avg']:>+6.2f}% "
          f"| {row['Risk_Class']}")

FACTORY RISK CLASSIFICATION
  Company Avg Margin: 65.96%

  Lot's O' Nuts        | Margin:  69.20% | Gap:  +3.24% | PERFORMING ✅
  Wicked Choccy's      | Margin:  65.12% | Gap:  -0.84% | UNDERPERFORMING ⚠️
  Sugar Shack          | Margin:  52.09% | Gap: -13.87% | UNDERPERFORMING ⚠️
  Secret Factory       | Margin:  51.71% | Gap: -14.25% | UNDERPERFORMING ⚠️
  The Other Factory    | Margin:  12.88% | Gap: -53.08% | CRITICAL RISK 🔴


### 6. Key Observations

In [8]:
best_factory  = factory_summary.iloc[0]
worst_factory = factory_summary.iloc[-1]

print("KEY OBSERVATIONS")
print(f"""
  🏆 Best Factory  : {best_factory['Factory']}
     Avg Margin    : {best_factory['Avg_Margin']}%
     Profit Share  : {best_factory['Profit_Share_%']}%
     Why it works  : Highest margin products, efficient cost structure

  🚨 Worst Factory : {worst_factory['Factory']}
     Avg Margin    : {worst_factory['Avg_Margin']}%
     Gap vs Avg    : {worst_factory['Margin_vs_Company_Avg']}% below company average
     Products      : {', '.join(product_factory[product_factory['Factory'] == worst_factory['Factory']]['Product Name'].tolist())}
     Risk          : Margins so low this factory adds negligible profit value

  💡 Core Insight  :
     The margin gap between best ({best_factory['Factory']}) and 
     worst ({worst_factory['Factory']}) factory is 
     {round(best_factory['Avg_Margin'] - worst_factory['Avg_Margin'], 2)} percentage points.
     This is a sourcing and cost negotiation problem, not a sales problem.
""")

KEY OBSERVATIONS

  🏆 Best Factory  : Lot's O' Nuts
     Avg Margin    : 69.2%
     Profit Share  : 56.78%
     Why it works  : Highest margin products, efficient cost structure

  🚨 Worst Factory : The Other Factory
     Avg Margin    : 12.88%
     Gap vs Avg    : -53.08% below company average
     Products      : Hair Toffee, Kazookles
     Risk          : Margins so low this factory adds negligible profit value

  💡 Core Insight  :
     The margin gap between best (Lot's O' Nuts) and 
     worst (The Other Factory) factory is 
     56.32 percentage points.
     This is a sourcing and cost negotiation problem, not a sales problem.



### Save Report

In [9]:
# Add coordinates to factory summary for Streamlit map later
factory_summary['Latitude']  = factory_summary['Factory'].map(
    lambda f: factory_coords[f]['latitude']
)
factory_summary['Longitude'] = factory_summary['Factory'].map(
    lambda f: factory_coords[f]['longitude']
)
factory_summary['State'] = factory_summary['Factory'].map(
    lambda f: factory_coords[f]['state']
)

factory_report = {
    "company_avg_margin"    : round(company_avg_margin, 2),
    "factory_summary"       : factory_summary.to_dict(orient='records'),
    "product_factory_detail": product_factory.to_dict(orient='records'),
    "factory_coords"        : factory_coords,
    "key_observations": {
        "best_factory"  : best_factory['Factory'],
        "worst_factory" : worst_factory['Factory'],
        "margin_gap"    : round(best_factory['Avg_Margin'] - worst_factory['Avg_Margin'], 2),
        "critical_finding": (
            f"{worst_factory['Factory']} operates at {worst_factory['Avg_Margin']}% margin "
            f"vs company average of {round(company_avg_margin, 2)}%. "
            f"This is a {round(company_avg_margin - worst_factory['Avg_Margin'], 2)} percentage point gap "
            f"indicating a serious cost structure problem that requires immediate attention."
        )
    }
}

report_path = '../outputs/reports/factory_analysis_report.json'
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(factory_report, f, indent=4)

print(f"Factory analysis report saved to: {report_path}")

Factory analysis report saved to: ../outputs/reports/factory_analysis_report.json
